# Our Smart PDF-to-Markdown Pipeline

Converting PDFs to Markdown is essential for our RAG chatbot because Markdown preserves structure (like headers, tables, and lists) in a way that language models understand easily.

Instead of treating every PDF the same, our system now uses a **Smart Conversion Workflow**. We analyze the PDF first and then route it to the best tool for the job. We classify PDFs into three simple categories:

1. **Scanned Documents**: If there's barely any selectable text, it's probably scanned. We need OCR (Optical Character Recognition) to read it.
2. **Image-Heavy Documents**: If a page has lots of charts or images, we need a Vision-Language Model (VLM) like Google Gemini to "look" at the page and describe the visuals.
3. **Simple Documents**: For standard digital text PDFs, we just extract the text quickly and directly.

Let's see how this works in practice!

## Workflow Diagram

```text
┌─────────────────────────────┐
│  Incoming PDF Document      │
└──────────┬──────────────────┘
           │
           ▼
┌─────────────────────────────┐
│  Quick Analysis (PyMuPDF)   │
│  - Check text density       │
│  - Count images             │
└──────────┬──────────────────┘
           │
           ▼
    ┌──────────────────┐
    │ Is it Scanned?   │───Yes──► Docling (OCR + Tables)
    │ (< 50 chars)     │
    └──────┬───────────┘
           │No
           ▼
    ┌────────────────────────────────┐
    │ Is it Image-Heavy?             │
    │ (> 2 images)                   │───Yes──► Gemini VLM 
    └──────┬─────────────────────────┘
           │No
           ▼
    ┌──────────────────────────────┐
    │ Simple Digital PDF           │───Yes──► PyMuPDF4LLM
    └──────────────────────────────┘
```

In [ ]:
# 1. Install our required packages
!pip install PyMuPDF pymupdf4llm docling google-genai

## Step 1: Analyzing the PDF

Before converting, we take a quick peek at the first page of the PDF to see what we're dealing with. We use `fitz` (PyMuPDF) to count characters and images.

In [ ]:
import fitz  # PyMuPDF

def analyze_pdf(file_path):
    doc = fitz.open(file_path)
    sample_page = doc[0] # Look at the first page
    
    # Check how much selectable text there is
    text = sample_page.get_text()
    is_scanned = len(text.strip()) < 50
    
    # Check how many images are embedded
    image_count = len(sample_page.get_images())
    has_images = image_count > 2
    
    doc.close()
    
    if is_scanned:
        return "SCANNED"
    elif has_images:
        return "IMAGE_HEAVY"
    else:
        return "SIMPLE"

# Example usage:
# pdf_type = analyze_pdf('my_document.pdf')
# print(f"This PDF is: {pdf_type}")

## Step 2: Processing Simple PDFs

If the document is a standard, text-heavy digital PDF, we use **PyMuPDF4LLM**. It's fast, accurate, and turns digital PDFs into Markdown instantly.

In [ ]:
import pymupdf4llm

def process_simple_pdf(file_path):
    print("Converting simple PDF...")
    md_text = pymupdf4llm.to_markdown(file_path)
    return md_text

## Step 3: Processing Scanned PDFs

If the document is scanned, standard text extraction fails. We use **Docling** with OCR and Table Structure Extraction turned on to read the text straight from the pixels.

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

def process_scanned_pdf(file_path):
    print("Converting scanned PDF with Docling OCR...")
    
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_table_structure = True  # Try to find tables
    pipeline_options.do_ocr = True              # Use OCR to read text
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    result = converter.convert(file_path)
    return result.document.export_to_markdown()

## Step 4: Processing Image-Heavy PDFs

If the document is full of charts, diagrams, or complex layouts, we use a Vision-Language Model (VLM) like **Google Gemini**. We turn the PDF page into an image and ask Gemini to write the markdown for us based on what it sees.

In [ ]:
from google import genai
from google.genai import types

def process_image_heavy_pdf(file_path, api_key):
    print("Converting complex PDF visually with Gemini...")
    client = genai.Client(api_key=api_key)
    
    doc = fitz.open(file_path)
    all_markdown = []
    
    for page_num in range(doc.page_count):
        page = doc[page_num]
        
        # Convert the PDF page to a high-res image
        pix = page.get_pixmap(matrix=fitz.Matrix(300/72, 300/72))
        img_data = pix.tobytes("png")
        image_part = types.Part.from_bytes(data=img_data, mime_type="image/png")
        
        # Ask Gemini to turn the image into markdown
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                "Convert this PDF page to clean, structured markdown. Extract all text, describe images, and preserve the layout.",
                image_part
            ],
        )
        all_markdown.append(f"# Page {page_num + 1}\n\n{response.text}")
        
    doc.close()
    return "\n\n---\n\n".join(all_markdown)

## Conclusion

By dynamically analyzing PDFs and routing them to the right tool, our chatbot ensures it always gets the highest quality Markdown data, leading to much better answers!